<a href="https://colab.research.google.com/github/gab-netize87724/Final-Project-CS2/blob/main/Dahlia_Attendance_NB_FA2%20(Updated).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
url = "https://raw.githubusercontent.com/gab-netize87724/Final-Project-CS2/refs/heads/main/attendance.json"
response = requests.get(url)
students = response.json()
print(students)

[{'student_id': 'S001', 'name': 'Alice Johnson', 'dates': [{'date': '2025-09-01', 'status': 'Present'}, {'date': '2025-09-02', 'status': 'Absent - Excused'}, {'date': '2025-09-03', 'status': 'Present'}, {'date': '2025-09-04', 'status': 'Tardy'}, {'date': '2025-09-05', 'status': 'Cut Class'}, {'date': '2025-09-06', 'status': 'Present'}]}, {'student_id': 'S002', 'name': 'Brian Smith', 'dates': [{'date': '2025-09-01', 'status': 'Tardy'}, {'date': '2025-09-02', 'status': 'Present'}, {'date': '2025-09-03', 'status': 'Absent - Unexcused'}, {'date': '2025-09-04', 'status': 'Present'}, {'date': '2025-09-05', 'status': 'Present'}, {'date': '2025-09-06', 'status': 'Cut Class'}]}, {'student_id': 'S003', 'name': 'Carla Gomez', 'dates': [{'date': '2025-09-01', 'status': 'Present'}, {'date': '2025-09-02', 'status': 'Present'}, {'date': '2025-09-03', 'status': 'Absent - Excused'}, {'date': '2025-09-04', 'status': 'Present'}, {'date': '2025-09-05', 'status': 'Tardy'}, {'date': '2025-09-06', 'status': 

In [ ]:
!pip install firebase-admin

In [ ]:
import firebase_admin
from firebase_admin import credentials, db
# Load the private key
cred = credentials.Certificate("/content/csfirebase-keynew.json")
# Initialize the app with your database URL
firebase_admin.initialize_app(cred, {
 "databaseURL": "https://dahlia-attendance-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
})
print("Firebase connected successfully!")

Firebase connected successfully!


In [ ]:
import requests
import json

data = students
print("JSON file loaded")
ref = db.reference("student")
for student in data:
  ref.child(str(student["student_id"])).set(student)
print("Data uploaded successfully!")

JSON file loaded
Data uploaded successfully!


In [ ]:
students_data = students_ref = db.reference("student").get()

for student_id, info in students_data.items():
 print(info['student_id'])
 print(info['dates'])

S001
[{'date': '2025-09-01', 'status': 'Present'}, {'date': '2025-09-02', 'status': 'Absent - Excused'}, {'date': '2025-09-03', 'status': 'Present'}, {'date': '2025-09-04', 'status': 'Tardy'}, {'date': '2025-09-05', 'status': 'Cut Class'}, {'date': '2025-09-06', 'status': 'Present'}]
S002
[{'date': '2025-09-01', 'status': 'Tardy'}, {'date': '2025-09-02', 'status': 'Present'}, {'date': '2025-09-03', 'status': 'Absent - Unexcused'}, {'date': '2025-09-04', 'status': 'Present'}, {'date': '2025-09-05', 'status': 'Present'}, {'date': '2025-09-06', 'status': 'Cut Class'}]
S003
[{'date': '2025-09-01', 'status': 'Present'}, {'date': '2025-09-02', 'status': 'Present'}, {'date': '2025-09-03', 'status': 'Absent - Excused'}, {'date': '2025-09-04', 'status': 'Present'}, {'date': '2025-09-05', 'status': 'Tardy'}, {'date': '2025-09-06', 'status': 'Absent - Unexcused'}]
S004
[{'date': '2025-09-01', 'status': 'Cut Class'}, {'date': '2025-09-02', 'status': 'Present'}, {'date': '2025-09-03', 'status': 'Pr

In [ ]:
for student_id, info in students_data.items():
  excused = 0
  unexcused = 0
  tardy = 0
  print(info['student_id'])
  for date in info['dates']:
    status = date['status']
    if status == ('Absent - Excused'):
      excused +=1
    elif status == ('Absent - Unexcused'):
      unexcused +=1
    elif status == ('Tardy'):
      tardy +=1
    elif status == ('Present'):
      continue
    else:
      unexcused +=1

  print("Excused Absences:", excused)
  print("Unexcused Absences:", unexcused)
  print("Tardy:", tardy)


S001
Excused Absences: 1
Unexcused Absences: 1
Tardy: 1
S002
Excused Absences: 0
Unexcused Absences: 2
Tardy: 1
S003
Excused Absences: 1
Unexcused Absences: 1
Tardy: 1
S004
Excused Absences: 0
Unexcused Absences: 2
Tardy: 1


In [ ]:
from firebase_admin import db

ref = db.reference("student")

while True:
    print("\n===== STUDENT DATABASE MENU =====")
    print("1. Display Students")
    print("2. Add Student")
    print("3. Update Student")
    print("4. Delete Student")
    print("5. Exit")

    choice = input("Enter choice: ")

    # DISPLAY
    if choice == "1":
       students_data = db.reference("student").get()
       if students_data:
         sorted_students = sorted(students_data.items(), key=lambda item: item[0])
         for student_id, info in sorted_students:
           print(f"ID: {info['student_id']}, Name: {info['name']}")
           if 'dates' in info:
             for date_entry in info['dates']:
               print(f"  Date: {date_entry['date']}, Status: {date_entry['status']}")
           else:
             print("No attendance dates recorded.")
       else:
         print("No students found in the database.")

    # ADD
    elif choice == "2":
        sid = input("Enter ID: ")
        if ref.child(sid).get():
          print("ID already exists! Try updating instead.")
          continue
        name = input("Enter name: ").upper()

        allowed_status = ["Present", "Absent - Excused", "Absent - Unexcused", "Tardy", "Cut Class"]

        dates_list = []
        try:
            num_dates = int(input("How many attendance dates to add? "))
            for i in range(num_dates):
                date_input = input(f"Enter date {i+1} (YYYY-MM-DD): ")
                while True:
                  status_input = input(f"Enter status for {date_input} (e.g., Present, Absent - Excused/Unexcused, Tardy, Cut Class): ")
                  if status_input in allowed_status:
                    formatted_status = status_input.title()
                    dates_list.append({"date": date_input, "status": formatted_status})
                    break
                  else:
                    print("Invalid status. Please enter a valid status.")
        except ValueError:
            print("Invalid input for number of dates. No dates will be added.")


        student = {
            "student_id": sid,
            "name": name,
            "dates": dates_list
        }

        ref.child(sid).set(student)
        print("Student added successfully!")

    # UPDATE
    elif choice == "3":
      sid = input("Enter ID of student to update: ")
      student_ref = db.reference("student").child(sid)
      student = student_ref.get()

      if student:
        name = input("Enter new name (leave blank to keep current): ")
        if name:
          student_ref.update({"name": name})
          print("Name updated!")

        if "dates" not in student or not student["dates"]:
            print("No attendance dates recorded. You can add new ones.")
            student["dates"] = []

        if student["dates"]:
            print("Current attendance dates:")
            for i, record in enumerate(student["dates"]):
                print(f"{i+1}. {record['date']} - {record['status']}")
            num_dates_input = input("How many dates do you want to update? (leave blank to skip): ")
            if num_dates_input:
                try:
                  num_dates = int(num_dates_input)
                  allowed_status = ["Present", "Absent - Excused", "Absent - Unexcused", "Tardy", "Cut Class"]
                  for _ in range(num_dates):
                    date_input = input("Enter date (YYYY-MM-DD): ")
                    found = False
                    for record in student.get("dates", []):
                      if record["date"] == date_input:
                        found = True
                        while True:
                          status_input = input(f"Enter status for {date_input}: ").title()
                          if status_input in allowed_status:
                            record["status"] = status_input.title()
                            student_ref.update({"dates": student["dates"]})
                            print(f"Status for {date_input} updated!")
                            break
                          else:
                            print("Invalid status. Please enter a valid status.")
                    if not found:
                      print(f"No record found for date {date_input}. Skipping update.")
                except ValueError:
                  print("Invalid input for number of dates. No dates will be updated.")
            print("Update complete!")
      else:
          print("Student not found.")

    # DELETE
    elif choice == "4":
      sid = input("Enter ID to delete: ")
      if db.reference("student").child(sid).get():
        db.reference("student").child(sid).delete()
        print("Student deleted successfully!")
      else:
        print("Student not found.")

    # EXIT
    elif choice == "5":
      print("Exiting program...")
      break
    else:
      print("Invalid choice. Try again.")


===== STUDENT DATABASE MENU =====
1. Display Students
2. Add Student
3. Update Student
4. Delete Student
5. Exit
Enter choice: 5
Exiting program...
